# NLP-Driven Dynamic Product Feed Optimization

### Case Study

#### Semantic Product–Trend Matching using Sentence Transformers and FAISS

---

### Objective

The goal of this project is to build a semantic matching engine that connects real-time social media trends with products from an e-commerce catalog.

Traditional keyword matching often fails to capture consumer intent because trending phrases and product descriptions may use different vocabulary while referring to similar concepts. To address this challenge, this project leverages transformer-based sentence embeddings and vector similarity search to identify semantically relevant product-trend pairs.

The final output is an optimized product catalog where each product is associated with its most relevant social media trends and an automatically generated optimized product title.

## Table of Contents

1. Project Overview
2. Exploratory Data Analysis
3. Data Preprocessing
4. Embedding Generation
5. Similarity Search with FAISS
6. Trend Matching
7. Product Title Optimization
8. Results
9. Conclusion

In [ ]:
!pip install sentence-transformers faiss-cpu pandas numpy scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
import faiss

In [ ]:
trends = pd.read_csv("social_media_trends_large.csv")

products = pd.read_csv("ecommerce_product_catalog_large.csv")

In [ ]:
print("Trends Dataset Shape:", trends.shape)

print("Products Dataset Shape:", products.shape)

In [ ]:
print("TRENDS DATASET")
trends.info()

In [ ]:
print("PRODUCTS DATASET")
products.info()

In [ ]:
trends.head()

In [ ]:
products.head()

In [ ]:
print("Missing Values in Trends Dataset")
print(trends.isnull().sum())

In [ ]:
print("\nMissing Values in Products Dataset")
print(products.isnull().sum())

In [ ]:
print("Duplicate Trend Rows:",
      trends.duplicated().sum())

print("Duplicate Product Rows:",
      products.duplicated().sum())

# Exploratory Data Analysis (EDA)

The purpose of EDA is to understand the structure, quality, and characteristics of both datasets before applying NLP techniques.

The analysis includes:

- Dataset dimensions
- Missing value analysis
- Duplicate record analysis
- Trend distribution across platforms
- Product category distribution
- Search volume analysis
- Product pricing analysis
- Text length analysis

In [ ]:
trends["platform_source"].value_counts().plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Platform Source Distribution")
plt.xlabel("Platform")
plt.ylabel("Count")

plt.show()

In [ ]:
trends["trend_category"].value_counts().head(10).plot(
    kind="bar",
    figsize=(10,5)
)

plt.title("Top 10 Trend Categories")

plt.show()

In [ ]:
trends["search_volume_score"].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    trends["search_volume_score"],
    bins=30
)

plt.title("Search Volume Distribution")
plt.xlabel("Search Volume Score")
plt.ylabel("Frequency")

plt.show()

In [ ]:
products["category"].value_counts().head(10).plot(
    kind="bar",
    figsize=(10,5)
)

plt.title("Top Product Categories")

plt.show()

In [ ]:
products["price_inr"].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    products["price_inr"],
    bins=30
)

plt.title("Product Price Distribution")
plt.xlabel("Price (INR)")
plt.ylabel("Frequency")

plt.show()

In [ ]:
trends["trend_keyword"] = trends["trend_keyword"].fillna("")

products["product_description"] = products["product_description"].fillna("")

products["product_title"] = products["product_title"].fillna("")

In [ ]:
products["description_length"] = products[
    "product_description"
].apply(len)

products["description_length"].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    products["description_length"],
    bins=30
)

plt.title("Product Description Length Distribution")
plt.xlabel("Characters")
plt.ylabel("Count")

plt.show()

In [ ]:
trends["keyword_length"] = trends[
    "trend_keyword"
].apply(len)

trends["keyword_length"].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    trends["keyword_length"],
    bins=30
)

plt.title("Trend Keyword Length Distribution")
plt.xlabel("Characters")
plt.ylabel("Count")

plt.show()

# Data Preprocessing

Before generating embeddings, textual data must be cleaned and standardized.

The preprocessing steps include:

- Handling missing values
- Converting text to lowercase
- Removing special characters
- Removing unnecessary spaces
- Combining product titles and descriptions to create richer semantic representations

In [ ]:
def clean_text(text):

    text = str(text)

    text = text.lower()

    text = re.sub(
        r'[^a-zA-Z0-9\s]',
        '',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text

In [ ]:
trends["clean_trend"] = trends[
    "trend_keyword"
].apply(clean_text)

products["clean_description"] = products[
    "product_description"
].apply(clean_text)

In [ ]:
products["combined_text"] = (
    products["product_title"] + " " +
    products["product_description"]
)

products["combined_text"] = products[
    "combined_text"
].apply(clean_text)

# Semantic Embedding Generation

To capture the semantic meaning of trends and product information, a pre-trained Sentence Transformer model is used.

Model Selected:
- all-MiniLM-L6-v2

This model converts text into dense numerical vectors (embeddings) that preserve semantic relationships between phrases and sentences.

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
trend_embeddings = model.encode(
    trends["clean_trend"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

In [ ]:
product_embeddings = model.encode(
    products["combined_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

In [ ]:
print("Trend Embeddings Shape:")
print(trend_embeddings.shape)

print("\nProduct Embeddings Shape:")
print(product_embeddings.shape)

# Similarity Search using FAISS

Computing similarity between every trend and every product would require approximately:

25,000 × 50,000 = 1.25 Billion comparisons

To improve efficiency, Facebook AI Similarity Search (FAISS) is used.

FAISS enables fast nearest-neighbor retrieval without generating the complete similarity matrix in memory.

In [ ]:
faiss.normalize_L2(
    trend_embeddings
)

faiss.normalize_L2(
    product_embeddings
)

In [ ]:
dimension = trend_embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    trend_embeddings
)

print("Index Created Successfully")

In [ ]:
k = 3

similarity_scores, trend_indices = index.search(
    product_embeddings,
    k
)

# Trend Matching and Retrieval

For each product embedding, the FAISS index is queried to retrieve the Top-3 most semantically similar social media trends.

Cosine similarity is used as the similarity metric after embedding normalization.

In [ ]:
product_number = 0

print(
    products.iloc[product_number]
    ["product_description"]
)

print("\nTop Matches:\n")

for idx, score in zip(
    trend_indices[product_number],
    similarity_scores[product_number]
):

    print(
        trends.iloc[idx]["trend_keyword"],
        round(float(score),4)
    )

In [ ]:
top_trend_1 = []
top_trend_2 = []
top_trend_3 = []

score_1 = []
score_2 = []
score_3 = []

In [ ]:
for i in range(len(products)):

    idx1 = trend_indices[i][0]
    idx2 = trend_indices[i][1]
    idx3 = trend_indices[i][2]

    top_trend_1.append(
        trends.iloc[idx1]["trend_keyword"]
    )

    top_trend_2.append(
        trends.iloc[idx2]["trend_keyword"]
    )

    top_trend_3.append(
        trends.iloc[idx3]["trend_keyword"]
    )

    score_1.append(
        float(similarity_scores[i][0])
    )

    score_2.append(
        float(similarity_scores[i][1])
    )

    score_3.append(
        float(similarity_scores[i][2])
    )

In [ ]:
products["top_trend_1"] = top_trend_1
products["top_trend_2"] = top_trend_2
products["top_trend_3"] = top_trend_3

products["similarity_score_1"] = score_1
products["similarity_score_2"] = score_2
products["similarity_score_3"] = score_3

In [ ]:
SIMILARITY_THRESHOLD = 0.50

products["top_trend_1"] = np.where(
    products["similarity_score_1"] >= SIMILARITY_THRESHOLD,
    products["top_trend_1"],
    "No Strong Match"
)

# Dynamic Product Title Optimization

After identifying the most relevant trend for each product, an optimized product title is generated.

The objective is to naturally incorporate trending keywords into product titles, improving discoverability, relevance, and potential advertising performance.

In [ ]:
optimized_titles = []

for _, row in products.iterrows():

    trend = row["top_trend_1"]

    title = row["product_title"]

    if trend != "No Strong Match":

        optimized_title = (
            f"{trend} Inspired {title}"
        )

    else:

        optimized_title = title

    optimized_titles.append(
        optimized_title
    )

In [ ]:
products[
    "optimized_product_title"
] = optimized_titles

In [ ]:
products[
    [
        "product_title",
        "top_trend_1",
        "similarity_score_1",
        "optimized_product_title"
    ]
].head(10)

# Final Output Generation

The enriched product catalog now contains:

- Top 3 matching trends
- Similarity scores
- Optimized product titles

The final dataset is exported as a CSV file for further business use and deployment.

In [ ]:
sample_output = products.head(500)

sample_output.to_csv(
    "sample_optimized_catalog.csv",
    index=False
)

from google.colab import files

files.download("sample_optimized_catalog.csv")

# Conclusion

This project successfully demonstrates the application of Natural Language Processing (NLP) and vector similarity search for dynamic product feed optimization.

Key achievements:

- Preprocessed and analyzed large-scale trend and product datasets.
- Generated semantic embeddings using Sentence Transformers.
- Built an efficient similarity search engine using FAISS.
- Identified the Top-3 relevant trends for each product.
- Generated optimized product titles based on trend relevance.

By leveraging semantic matching instead of traditional keyword matching, the solution better captures consumer intent and enables more effective alignment between social demand and product inventory.